# 04 — Shunting and parking calibration

Per-event facility charges, written to `data/facility_charges.csv`. Narrative
in `SHUNTING_PARKING.md`.

**Why this notebook is shaped differently from `02`.** The raw tariffs span a
factor of 150 for what is physically the same operation, and that spread is
*scope*, not price level: Hungary bundles the shunting locomotive into its
tariff, Germany sells only the track, Greece states explicitly that its charge
covers the manoeuvring team but not the locomotive. Calibrating the IM tariff
alone understates the operator's cost roughly fourfold. So the model is
two-component — IM tariff plus a market top-up for whatever the IM does not
supply — and that top-up is the largest assumption in the file.

In [ ]:
# STDLIB-ONLY cell. See the seed-export contract in calib/README.md.
import csv
from pathlib import Path


def _calib_dir() -> Path:
    """Anchor on the calib folder whatever the kernel's cwd happens to be."""
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if (cand / "resolution.py").exists():
            return cand
    for sub in ("backend/models/infrastructure/calib", "models/infrastructure/calib"):
        if (cwd / sub).is_dir():
            return (cwd / sub).resolve()
    raise RuntimeError("cannot locate models/infrastructure/calib")


CALIB_DIR = _calib_dir()
DATA_DIR = CALIB_DIR / "facility_calibration" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def write_data(name: str, fieldnames: list[str], rows: list[dict]) -> None:
    """Write one committed observation table to calib/facility_calibration/data/."""
    with open(DATA_DIR / name, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f"facility_calibration/data/{name}: {len(rows)} rows")


def read_data(name: str) -> list[dict]:
    with open(DATA_DIR / name, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

In [ ]:
# STDLIB-ONLY cell.
from dataclasses import dataclass, asdict
from typing import Optional

# How much evidence stands behind a value. Not a quality judgement — a
# well-argued ASSUMED and a mis-transcribed SOURCED are both possible; the
# status says which kind of thing the reader is looking at.
SOURCED = "sourced"  # named document, named locator
NOT_LEVIED = "not_levied"  # positively documented as zero — not absent data
DERIVED = "derived"  # arithmetic on other values, formula in the note
BENCHMARK = "benchmark"  # pan-European statistic standing in for a country
ASSUMED = "assumed"  # judgement, with a mandatory low/high band
MISSING = "missing"  # nothing read yet — never an estimate
NO_RAILWAY = "no_railway"

USABLE = {SOURCED, NOT_LEVIED, DERIVED, BENCHMARK, ASSUMED}


@dataclass(frozen=True)
class SV:
    """
    One calibrated parameter with its audit trail.

    source_id points at sources_register.csv; locator is what makes it
    re-checkable a year later (section, table, sheet), because a network
    statement runs to hundreds of pages and its tariff tables move between
    editions. Values stay in native currency and price basis — conversion and
    escalation happen later and explicitly.
    """

    country_code: str
    parameter: str
    value: Optional[float]
    unit: str
    status: str
    source_id: str = ""
    locator: str = ""
    currency: str = "EUR"
    basis_year: Optional[int] = None
    note: str = ""
    low: Optional[float] = None  # sensitivity band, mandatory when ASSUMED
    high: Optional[float] = None

    def __post_init__(self):
        if self.status in USABLE and self.value is None:
            raise ValueError(
                f"{self.country_code}/{self.parameter}: {self.status} needs a value"
            )
        if self.status in (SOURCED, NOT_LEVIED) and not (
            self.source_id and self.locator
        ):
            raise ValueError(
                f"{self.country_code}/{self.parameter}: sourced needs source_id + locator"
            )
        if self.status == ASSUMED and (
            self.low is None or self.high is None or not self.note
        ):
            raise ValueError(
                f"{self.country_code}/{self.parameter}: assumed needs a band and a rationale"
            )
        if self.status == DERIVED and not self.note:
            raise ValueError(
                f"{self.country_code}/{self.parameter}: derived must state its formula"
            )


SV_FIELDS = [
    "country_code",
    "parameter",
    "value",
    "unit",
    "status",
    "source_id",
    "locator",
    "currency",
    "basis_year",
    "note",
    "low",
    "high",
]


def emit(name: str, values: list[SV]) -> None:
    write_data(name, SV_FIELDS, [asdict(v) for v in values])
    by_status: dict[str, int] = {}
    for v in values:
        by_status[v.status] = by_status.get(v.status, 0) + 1
    for s, n in sorted(by_status.items(), key=lambda kv: -kv[1]):
        print(f"    {s:11} {n:4}")

## Scope classes, labour index, reference rotation

Top-ups at index 1.00: a locomotive is ~110 EUR/event, locomotive plus crew
~190. The index band comes from Ramboll's driver-hour rates of 62–104 EUR/h
across fifteen countries — a 1.68x spread, far narrower than the tariffs
suggest, which is the point.

Parking is deliberately **not** indexed: the four sourced length-based rates
show no wage signal at all, because stabling is land and track rather than
labour.

In [ ]:
FULL, CREW, TRACK = "full", "crew", "track"
TOPUP = {FULL: 0.0, CREW: 110.0, TRACK: 190.0}
REF_LENGTH_M, REF_HOURS = 300.0, 12.0

LABOUR_INDEX = {
    **{
        cc: 1.25
        for cc in (
            "AT",
            "BE",
            "CH",
            "DE",
            "DK",
            "FI",
            "FR",
            "IE",
            "LU",
            "NL",
            "NO",
            "SE",
            "UK",
        )
    },
    **{cc: 1.00 for cc in ("CZ", "EE", "ES", "IT", "PT", "SI")},
    **{cc: 0.75 for cc in ("BG", "GR", "HR", "HU", "LT", "LV", "PL", "RO", "SK")},
}

BASIS = [
    SV(
        "--",
        "market_topup_loco_crew",
        190.0,
        "EUR/event",
        ASSUMED,
        "NOX-MODEL",
        "line 213",
        "EUR",
        2030,
        "One hour of shunting locomotive plus crew bought on the market. Calibrated so the German "
        "rotation reproduces the nox model's OPS Infrastructure Access line (307.55 EUR/trip) and "
        "cross-checked against the three full-scope sourced tariffs (AT 99, ES 148, HU 263; mean ~170).",
        low=120.0,
        high=260.0,
    ),
    SV(
        "--",
        "market_topup_loco_only",
        110.0,
        "EUR/event",
        ASSUMED,
        "NOX-MODEL",
        "line 213",
        "EUR",
        2030,
        "Locomotive alone, where the IM already supplies the crew (GR, PT).",
        low=70.0,
        high=160.0,
    ),
    SV(
        "--",
        "labour_index_band",
        1.0,
        "index",
        ASSUMED,
        "RAMBOLL-BMDV",
        "slide 6",
        "EUR",
        2025,
        "Three-tier index anchored on the driver-hour band of 62-104 EUR/h (1.68x spread, midpoint "
        "83). Applied to the market top-up only.",
        low=0.75,
        high=1.25,
    ),
]
print(f"{len(LABOUR_INDEX)} countries indexed")

## Sourced IM tariffs and parking bases

Ten countries publish a usable shunting figure; the rest take a `track`-scope
default of 10 EUR, the level observed where an IM sells only the facility
(DE 10.80, SI 13.27, PL 1.72). Since the top-up dominates those totals, the
default moves the answer by under 5 %.

Free allowances on parking are material and country-specific: Denmark levies
nothing, Norway's first 48 h are free, Croatia's first 24 h on a side track,
and Slovenia charges only unplanned storage.

In [ ]:
SHUNT = {
    "AT": (
        FULL,
        SV(
            "AT",
            "shunting_im",
            99.25,
            "EUR/event",
            DERIVED,
            "AT-SNNB-2026",
            "Tab.44",
            "EUR",
            2026,
            "39.70 EUR/h shunting leader with loco operation x 5 h minimum deployment, "
            "shared across the two moves of one turnaround",
        ),
    ),
    "BE": (
        TRACK,
        SV(
            "BE",
            "shunting_im",
            39.004714,
            "EUR/event",
            SOURCED,
            "BE-NS-2027",
            "App F.2 sheet 3.1.1.1",
            "EUR",
            2026,
            "unit cost per access to a service facility",
        ),
    ),
    "BG": (
        TRACK,
        SV(
            "BG",
            "shunting_im",
            2.20,
            "EUR/event",
            SOURCED,
            "BG-NRIC-2026",
            "§2.6.2",
            "EUR",
            2026,
            "0.20 EUR per draw-out/marshalling track use, assumed per vehicle x 11",
        ),
    ),
    "DE": (
        TRACK,
        SV(
            "DE",
            "shunting_im",
            10.80,
            "EUR/event",
            SOURCED,
            "DE-APS-2027",
            "§2.1 Zugbildung I",
            "EUR",
            2027,
            "1 h on a train-formation object; the loco and crew are a separate market. Facilities "
            "with an Anlagendisponent escalate steeply — 722.40 EUR for a 12 h stay.",
        ),
    ),
    "DK": (
        TRACK,
        SV(
            "DK",
            "shunting_im",
            0.0,
            "EUR/event",
            NOT_LEVIED,
            "DK-NS-2027",
            "§3.5",
            "EUR",
            2027,
            "no infrastructure charges for operations or parking on sidings",
        ),
    ),
    "ES": (
        FULL,
        SV(
            "ES",
            "shunting_im",
            148.00,
            "EUR/event",
            SOURCED,
            "ES-ADIF-2027",
            "ch.6 basic operations",
            "EUR",
            2027,
            "shunting driving operations 148 EUR/h; overall shunting operations 200 EUR/h",
        ),
    ),
    "GR": (
        CREW,
        SV(
            "GR",
            "shunting_im",
            30.00,
            "EUR/event",
            SOURCED,
            "GR-OSE-2026",
            "§6.4.1",
            "EUR",
            2019,
            "per manoeuvre, passenger; covers the manoeuvring team but not loco or driver",
        ),
    ),
    "HU": (
        FULL,
        SV(
            "HU",
            "shunting_im",
            263.00,
            "EUR/event",
            DERIVED,
            "HU-NS-2627",
            "Annex 5.2-6",
            "EUR",
            2027,
            "24,480 HUF/person/h staff + 79,752 HUF/vehicle/h traction unit, 1 h, "
            "at 396 HUF/EUR — the only IM that prices the shunting locomotive itself",
        ),
    ),
    "PL": (
        TRACK,
        SV(
            "PL",
            "shunting_im",
            1.72,
            "EUR/event",
            ASSUMED,
            "PL-PLK-A91",
            "Annex 9.1",
            "EUR",
            2027,
            "3.66 PLN/km electric traction x an assumed 2 km per movement. Actual "
            "distances are tabulated per location in Appendix 2.8; the total is under 2% sensitive.",
            low=0.9,
            high=4.6,
        ),
    ),
    "PT": (
        CREW,
        SV(
            "PT",
            "shunting_im",
            24.59,
            "EUR/event",
            SOURCED,
            "PT-IP-2027",
            "§5.4.4",
            "EUR",
            2027,
            "long duration >30 min; short duration is 9.75 EUR",
        ),
    ),
    "SI": (
        TRACK,
        SV(
            "SI",
            "shunting_im",
            13.27,
            "EUR/event",
            SOURCED,
            "SI-NS-2027",
            "§5.4.2 P22",
            "EUR",
            2027,
            "per departure from and arrival at the station of origin or a yard",
        ),
    ),
}

PARK = {
    "AT": dict(
        rate=SV(
            "AT",
            "parking_rate",
            0.150,
            "EUR/m/day",
            SOURCED,
            "AT-SNNB-2026",
            "Tab.49 item 4.2.2",
            "EUR",
            2026,
            "4.57 EUR/m/month on a long-term booking, the realistic mode for a daily rotation",
        )
    ),
    "BG": dict(
        rate=SV(
            "BG",
            "parking_rate",
            0.20,
            "EUR/m/day",
            SOURCED,
            "BG-NRIC-2026",
            "§12",
            "EUR",
            2026,
            "per metre of length per 24 h",
        )
    ),
    "NO": dict(
        free_h=48.0,
        rate=SV(
            "NO",
            "parking_rate",
            0.123,
            "EUR/m/day",
            SOURCED,
            "NO-NS-2027",
            "Tab.9 §7.3.5",
            "EUR",
            2026,
            "6 NOK/h per commenced 100 m outside Alnabru",
        ),
    ),
    "HR": dict(
        free_h=24.0,
        rate=SV(
            "HR",
            "parking_rate",
            0.053,
            "EUR/m/day",
            SOURCED,
            "HR-NS-2027",
            "§7.3.4.4 note 9",
            "EUR",
            2027,
            ">48 h band, establishment category 3",
        ),
    ),
    "DE": dict(
        flat=SV(
            "DE",
            "parking_per_hour",
            6.02,
            "EUR/h",
            SOURCED,
            "DE-APS-2027",
            "§2.1 Abstellung I",
            "EUR",
            2027,
            "length-independent by design — the APS calls the disponent charge expressly "
            "zuglaengenunabhaengig",
        )
    ),
    "PT": dict(
        free_h=1.0,
        flat=SV(
            "PT",
            "parking_per_hour",
            2.43,
            "EUR/h",
            DERIVED,
            "PT-IP-2027",
            "§5.4.4",
            "EUR",
            2027,
            "Te = 0.0405 EUR/min x 60; applies beyond the first hour, "
            "and timetabled technical stops are exempt",
        ),
    ),
    "IT": dict(
        event=SV(
            "IT",
            "parking_per_event",
            9.992,
            "EUR/event",
            SOURCED,
            "IT-NS-2027",
            "§5.4.6.1",
            "EUR",
            2027,
            "indirect cost per parking operation; the energy element belongs to notebook 03",
        )
    ),
    "GR": dict(
        event=SV(
            "GR",
            "parking_per_event",
            60.00,
            "EUR/event",
            DERIVED,
            "GR-OSE-2026",
            "§6.4.2",
            "EUR",
            2019,
            "stabling is defined as exactly 2 manoeuvres x 30 EUR — no time or length term",
        )
    ),
    "DK": dict(
        event=SV(
            "DK",
            "parking_per_event",
            0.0,
            "EUR/event",
            NOT_LEVIED,
            "DK-NS-2027",
            "§3.5",
            "EUR",
            2027,
            "no siding charges",
        )
    ),
    "SI": dict(
        event=SV(
            "SI",
            "parking_per_event",
            0.0,
            "EUR/event",
            NOT_LEVIED,
            "SI-NS-2027",
            "§5.4.3 P23",
            "EUR",
            2027,
            "planned storage not charged; P23 covers unplanned "
            "RU-attributable storage only",
        )
    ),
    "BE": dict(
        event=SV(
            "BE",
            "parking_per_event",
            0.0,
            "EUR/event",
            NOT_LEVIED,
            "BE-NS-2027",
            "App F.2 sheet 3.1.2.2",
            "EUR",
            2026,
            "occupancy charged only in yards declared congested",
        )
    ),
}

DEFAULT_SHUNT_IM = 10.0
DEFAULT_PARK_RATE = 0.20
BASIS += [
    SV(
        "--",
        "default_shunting_im",
        DEFAULT_SHUNT_IM,
        "EUR/event",
        ASSUMED,
        "",
        "",
        "EUR",
        2027,
        "Track-access-only level observed where the IM sells the facility rather than the service "
        "(DE 10.80, SI 13.27, PL 1.72). Used where no facility price list has been read.",
        low=0.0,
        high=40.0,
    ),
    SV(
        "--",
        "default_parking_rate",
        DEFAULT_PARK_RATE,
        "EUR/m/day",
        ASSUMED,
        "",
        "",
        "EUR",
        2027,
        "Median of the four sourced length-based tariffs (AT 0.150, BG 0.20, NO 0.123, HR 0.053). "
        "Gives 60 EUR/day for the 300 m reference train, consistent with the sourced per-event values.",
        low=0.05,
        high=0.35,
    ),
]
print(f"{len(SHUNT)} sourced shunting, {len(PARK)} sourced parking bases")

In [ ]:
VALUES: list[SV] = list(BASIS)
OUT: list[dict] = []

for cc in sorted(LABOUR_INDEX):
    scope, im = SHUNT.get(
        cc,
        (
            TRACK,
            SV(
                cc,
                "shunting_im",
                DEFAULT_SHUNT_IM,
                "EUR/event",
                ASSUMED,
                "",
                "",
                "EUR",
                2027,
                "track-scope default, see BASIS",
                low=0.0,
                high=40.0,
            ),
        ),
    )
    idx = LABOUR_INDEX[cc]
    topup = TOPUP[scope] * idx
    shunt = im.value + topup
    VALUES.append(im)
    VALUES.append(
        SV(
            cc,
            "shunting_allin",
            round(shunt, 2),
            "EUR/event",
            DERIVED,
            im.source_id,
            im.locator,
            "EUR",
            im.basis_year,
            f"IM tariff {im.value:.2f} + {scope} top-up {TOPUP[scope]:.0f} x index {idx:.2f}",
        )
    )

    spec = PARK.get(cc, {})
    free_h = spec.get("free_h", 0.0)
    billable = max(0.0, REF_HOURS - free_h)
    if "event" in spec:
        pv = spec["event"]
        park = pv.value
        basis = "per_event"
    elif "flat" in spec:
        pv = spec["flat"]
        park = pv.value * billable
        basis = "per_hour"
    else:
        pv = spec.get(
            "rate",
            SV(
                cc,
                "parking_rate",
                DEFAULT_PARK_RATE,
                "EUR/m/day",
                ASSUMED,
                "",
                "",
                "EUR",
                2027,
                "median default, see BASIS",
                low=0.05,
                high=0.35,
            ),
        )
        days = 0.0 if billable == 0 else max(1.0, -(-billable // 24))
        park = pv.value * REF_LENGTH_M * days
        basis = "per_metre_day"
    VALUES.append(pv)
    VALUES.append(
        SV(
            cc,
            "parking_per_event_ref",
            round(park, 2),
            "EUR/event",
            DERIVED,
            pv.source_id,
            pv.locator,
            "EUR",
            pv.basis_year,
            f"{basis} basis, {REF_LENGTH_M:.0f} m x {REF_HOURS:.0f} h, "
            f"{free_h:.0f} h free allowance",
        )
    )

    OUT.append(
        dict(
            country_code=cc,
            scope=scope,
            labour_index=idx,
            shunting_im_eur_event=round(im.value, 2),
            shunting_allin_eur_event=round(shunt, 2),
            parking_basis=basis,
            parking_free_hours=free_h,
            parking_eur_event_ref=round(park, 2),
            turnaround_eur=round(2 * shunt + park, 2),
        )
    )

write_data("facility_reference_rotation.csv", list(OUT[0]), OUT)
emit("facility_charges.csv", VALUES)

## Cross-check against the operator model

The nox model books 307.55 EUR per trip for infrastructure access. A turnaround
serves both the inbound and the outbound trip, so the comparable figure is half
a turnaround. Agreement within ~10 % is the evidence the top-up is the right
order of magnitude; the IM tariffs alone would be about a quarter of it.

In [ ]:
NOX_PER_TRIP = 307.55
de = next(r for r in OUT if r["country_code"] == "DE")
modelled = de["turnaround_eur"] / 2
im_only = (2 * de["shunting_im_eur_event"] + de["parking_eur_event_ref"]) / 2
print(f"DE modelled per trip : {modelled:7.2f} EUR")
print(
    f"nox model per trip   : {NOX_PER_TRIP:7.2f} EUR   deviation {modelled / NOX_PER_TRIP - 1:+.1%}"
)
print(
    f"IM tariffs alone     : {im_only:7.2f} EUR   ({im_only / NOX_PER_TRIP:.0%} of the operator figure)"
)
assert abs(modelled / NOX_PER_TRIP - 1) < 0.25, (
    "top-up calibration drifted from the operator model"
)

In [ ]:
# Display / validation only — pandas is fine here, seed.py skips this cell.
import pandas as pd

_df = pd.DataFrame([asdict(v) for v in VALUES])
_reg = set(pd.read_csv(DATA_DIR / "sources_register.csv")["source_id"])
_unknown = set(_df.loc[_df.source_id.ne(""), "source_id"]) - _reg
assert not _unknown, f"unregistered source ids: {sorted(_unknown)}"
_bad = _df[(_df.status == "assumed") & (_df.low.isna() | _df.high.isna())]
assert _bad.empty, _bad
print(f"{len(_df)} values, {_df.source_id.ne('').sum()} with a source pointer")
pd.DataFrame(OUT).sort_values("turnaround_eur")